# ICCD753 Recuperación de Información - Examen Final
**Tema:** Diseño e Implementación de un Sistema de Recuperación de Información (RAG)
**Estudiante:** [Nombre del Estudiante]

---
### Enlace de Despliegue en la Nube
**URL Streamlit Cloud:** [https://share.streamlit.io/...](https://share.streamlit.io/...)
---


## A. Preparación del corpus
Descargamos y estructuramos la base de datos `arXiv Paper Abstracts` usando Kaggle y Pandas.

In [ ]:
## B. Representación mediante embeddings
Inicializamos el modelo local de HuggingFace (all-MiniLM-L6-v2) para obtener representaciones vectoriales del texto.

from src.embeddings import TextEmbedder

embedder = TextEmbedder()
test_embedding = embedder.embed_text("Graph Neural Networks")
print(f"Dimensión del embedding: {len(test_embedding[0])}")

In [ ]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

load_dotenv()
embedder = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
test_embedding = embedder.embed_query("Graph Neural Networks")
print(f"Dimensión del embedding: {len(test_embedding)}")

## C. Almacenamiento y búsqueda vectorial
Se utiliza `ChromaDB` para almacenar de manera persistente los documentos y sus correspondientes vectores semánticos.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="data/chroma_db")
collection = client.get_or_create_collection(
    name="arxiv_corpus",
    metadata={"hnsw:space": "cosine"}
)
print(f"Documentos indexados actualmente: {collection.count()}")

## D. Recuperación
La clase `TextRetriever` encapsula la lógica para transformar la consulta a vector y extraer el Top-K desde ChromaDB.

In [ ]:
from src.retrieval import TextRetriever

retriever = TextRetriever()
results = retriever.retrieve("What are the main applications of Graph Neural Networks?", top_k=3)
for r in results:
    print(f"Score: {r['score']:.4f} | Title: {r['title']}")

## E. Generación aumentada por recuperación
A través de LangChain y Gemini se configura un *Prompt Estricto* para evitar alucinaciones y obligar al modelo a usar exclusivamente el contexto.

In [ ]:
from src.generation import RAGGenerator

generator = RAGGenerator()
answer = generator.generate_response(
    "What are the main applications of Graph Neural Networks?", 
    results
)
print("Respuesta Generada:\n", answer)

## F. Presentación de evidencias e Interfaz (G y H)
Toda esta lógica fue abstraída y construida dentro del script `src/app.py`. Para ejecutar la interfaz de Chat, se debe ejecutar:
```bash
streamlit run src/app.py
```
El sistema muestra debajo de cada respuesta un bloque colapsable (`st.expander`) con el título y resumen original del artículo (evidencia).

## I. Evaluación del sistema y de la generación
A continuación, se documenta el juicio subjetivo sobre el desempeño del sistema RAG tras realizar pruebas manuales con distintas consultas:

- **Corrección de la respuesta:** El modelo Gemini-2.5-flash produce respuestas altamente correctas semántica y gramaticalmente. Al establecer la temperatura en `0.0`, el modelo se restringe a responder de forma determinista y precisa.
- **Relevancia con respecto a la consulta:** Al usar embeddings de Google (`text-embedding-004`) y la métrica de similitud del coseno en ChromaDB, el sistema recupera satisfactoriamente los artículos que abordan el núcleo semántico de la pregunta del usuario.
- **Fidelidad respecto de las evidencias recuperadas:** El *prompt estricto* diseñado obliga al LLM a no utilizar conocimiento previo, forzando a que las afirmaciones generadas se correspondan uno a uno con el texto de las evidencias presentadas en pantalla.
- **Capacidad para integrar información de varios documentos:** El modelo es capaz de leer el contexto concatenado de los fragmentos Top-K, sintetizando ideas (por ejemplo, si dos papers hablan de grafos, agrupa las aplicaciones mencionadas en ambos en una única lista coherente).
- **Capacidad para reconocer cuando el corpus no contiene información suficiente:** Gracias a las instrucciones condicionadas en el prompt (regla 2), si se le pregunta por una receta de cocina o un tema ajeno a arXiv, el sistema responde explícitamente: *"el corpus no contiene información suficiente para responder la consulta"*, cumpliendo con éxito la mitigación de alucinaciones.

## J. Evaluación Cuantitativa (Opcional)
Adicionalmente, el sistema cuenta con los scripts `src/evaluate.py` y `src/evaluation_metrics.py` que calculan de forma programática el desempeño del sistema utilizando las métricas estándar de Recuperación de Información (**Precision@k, Recall@k y NDCG@k**).
Para probarlo, simplemente se ejecuta el siguiente comando en consola:
```bash
python src/evaluate.py
```
Esto evaluará las consultas de prueba contra el Ground Truth almacenado en `data/evaluation/qrels.json` y generará un reporte global de las métricas.